# 实验八 · 并发链表：读多写少场景的锁策略

**所属**：《并行计算》第四章 · Pthread 多线程编程　|　**难度**：⭐⭐⭐⭐ 综合　|　**预计时长**：30–40 分钟

> **实验说明**
> 1. 本章前几个实验处理的都是**规则的数组**：数据划分清晰，线程各写各的区间。本实验转向**动态数据结构**——一个有序链表，支持查找、插入、删除。操作会改变结构本身，且查找往往占绝大多数。
> 2. 同一个链表用四种方式保护：串行基准、一把大锁、逐节点交接锁、读写锁。四者的对比维度是**正确性、并发度、锁操作次数、实现复杂度**。
> 3. 本实验的程序由两个源文件组成：链表主程序，以及一个线程安全的随机数生成器 `pcg32`。请先写入随机数模块，再逐版本编写主程序。
> 4. 请自上而下依次执行各单元格（Shift+Enter）。
> 5. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明动态数据结构的并发保护为何比数组更困难
- 用一把全局锁实现正确的并发链表，并说明它为何几乎没有并发度
- 说明**交接锁**（hand-over-hand locking）的加锁规则，并解释它为何不会死锁
- 分析交接锁的锁操作次数为何正比于链表长度，从而认识其适用边界
- 使用**读写锁**区分读者与写者，说明它在读多写少场景下的优势
- 根据操作组成（读写比例）与数据规模，为并发数据结构选择合适的锁策略
- 说明为何 `rand()` 不能用于多线程，以及每线程独立随机数流的实现方式

## 🗺️ 学习路径

1. **准备阶段**：理解有序链表的三种操作，认识动态结构相较数组的新困难
2. **随机数模块**：为何 `rand()` 不可用，如何为每个线程配备独立的随机数流
3. **版本一 · 串行基准**：单线程，无锁，作为性能与正确性的基准
4. **版本二 · 一把大锁**：正确但完全串行化，多线程毫无收益
5. **版本三 · 交接锁**：逐节点加锁，提高并发度，但锁操作次数正比于链表长度
6. **版本四 · 读写锁**：读者共享、写者独占，读多写少时并发度接近理想
7. **性能对比**：在不同读写比例下比较四者，得出选择依据

## 1. 背景与动机

前面几个实验的并发对象都是**数组**：矩阵、向量、缓冲区。数组有一个便利的性质——**元素的位置是固定的**，可以按下标把数据划分给各线程，线程之间互不干涉。

**动态数据结构**没有这个便利。以有序链表为例：

- 节点在堆上动态分配，位置不固定；
- 一次插入或删除会**改变结构本身**——修改指针、增删节点；
- 一个操作需要**遍历**多个节点才能完成，遍历路径上的任何节点都可能正被另一个线程修改。

这带来数组所没有的风险。设想线程 A 正沿链表查找，而线程 B 恰好删除了 A 即将访问的节点并释放了它——A 随即解引用一个已经释放的指针，程序崩溃。

> 数组的并发是「**划分**」问题：把数据分给线程即可。
> 动态结构的并发是「**保护**」问题：必须在操作进行时，保证结构的相关部分不被他人破坏。

本实验以有序链表为载体，探讨这一保护应当做到多细的粒度。

## 2. 问题模型：有序链表与三种操作

一个**升序排列**的单向链表，节点定义为：

```c
struct list_node_s {
  int data;
  struct list_node_s *next;
};
struct list_node_s *head = NULL;
```

支持三种操作：

<!--
| 操作 | 语义 | 是否修改结构 |
|---|---|---|
| `Member(v)` | 查找 `v` 是否存在 | **否**（只读遍历） |
| `Insert(v)` | 插入 `v`，保持有序；已存在则不插 | 是（改指针、增节点） |
| `Delete(v)` | 删除 `v`；不存在则不动 | 是（改指针、删节点） |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">操作</th>
      <th style="text-align: left;">语义</th>
      <th style="text-align: left;">是否修改结构</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>Member(v)</code></td>
      <td style="text-align: left;">查找 <code>v</code> 是否存在</td>
      <td style="text-align: left;"><strong>否</strong>（只读遍历）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>Insert(v)</code></td>
      <td style="text-align: left;">插入 <code>v</code>，保持有序；已存在则不插</td>
      <td style="text-align: left;">是（改指针、增节点）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>Delete(v)</code></td>
      <td style="text-align: left;">删除 <code>v</code>；不存在则不动</td>
      <td style="text-align: left;">是（改指针、删节点）</td>
    </tr>
  </tbody>
</table>

**关键的负载特征**：真实系统中，**查找操作往往占绝大多数**。例如路由表、符号表、缓存索引，读远多于写。这一特征将直接决定哪种锁策略最优。

### 四个版本的对比维度

<!--
| 版本 | 锁策略 | 源文件后缀 |
|---|---|---|
| 一 · 串行基准 | 无锁（单线程） | `_base_seq` |
| 二 · 一把大锁 | 整个链表一把互斥量 | `_one_mutex` |
| 三 · 交接锁 | 每个节点一把互斥量 | `_node_mutex` |
| 四 · 读写锁 | 一把读写锁 | `_rwlock` |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">锁策略</th>
      <th style="text-align: left;">源文件后缀</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">一 · 串行基准</td>
      <td style="text-align: left;">无锁（单线程）</td>
      <td style="text-align: left;"><code>_base_seq</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">二 · 一把大锁</td>
      <td style="text-align: left;">整个链表一把互斥量</td>
      <td style="text-align: left;"><code>_one_mutex</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">三 · 交接锁</td>
      <td style="text-align: left;">每个节点一把互斥量</td>
      <td style="text-align: left;"><code>_node_mutex</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">四 · 读写锁</td>
      <td style="text-align: left;">一把读写锁</td>
      <td style="text-align: left;"><code>_rwlock</code></td>
    </tr>
  </tbody>
</table>

## 3. 环境准备

In [ ]:
import platform, subprocess, shutil, sys, os, re

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
NCPU = os.cpu_count()
print("CPU 核心:", NCPU)

if CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
elif NCPU == 1:
    print(
        "\n⚠️  当前仅 1 个核心：各版本无法真正并行，交接锁与读写锁的并发优势无法体现，"
    )
    print("    反而会因锁操作开销而显得更慢。本实验的性能结论必须在多核平台上验证。")
else:
    print(f"\n✅ 环境就绪：编译器可用，{NCPU} 核可用，可以开始实验！")


### 编译与运行工具函数

本实验的每个版本都由**两个源文件**编译而成：链表主程序 + 随机数模块 `pcg32_rand.c`。因此编译函数接受多个源文件。

In [2]:
SRC_DIR = "src_linkedlist"
os.makedirs(SRC_DIR, exist_ok=True)


def compile_c(out, *srcs):
    """编译一个或多个源文件为一个可执行文件。成功返回文件名，失败返回 None。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    src_str = " ".join(srcs)
    cmd = f"{base} -O3 -fPIC -pthread -Wall -Wextra {src_str} -o {out} -lpthread -lm"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("✅ 编译成功：", os.path.basename(out))
        if r.stderr.strip():
            print(r.stderr.strip())
        return out
    print("❌ 编译失败：\n", r.stderr)
    return None


def run_bin(out, *args, echo=True, timeout=300):
    """运行可执行文件并返回其标准输出；echo=True 时同时打印。"""
    r = subprocess.run(
        ["./" + out] + [str(a) for a in args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if echo:
        print(r.stdout, end="")
        if r.returncode != 0 and r.stderr:
            print("STDERR:", r.stderr)
    return r.stdout


def parse_throughput(text):
    """从输出中提取吞吐量（ops/sec）。"""
    m = re.search(r"Throughput\s*:\s*([\d.]+)", text)
    return float(m.group(1)) if m else None


def parse_size(text):
    """从输出中提取初始链表大小。"""
    m = re.search(r"Initial list size:\s*(\d+)", text)
    return int(m.group(1)) if m else None


## 4. 随机数模块：为什么不用 `rand()`

在测量并发数据结构之前，必须先解决一个基础问题：**如何在多线程中生成随机数**。

标准库的 `rand()` 有两个问题，使它不能用于本实验：

1. **非线程安全**。`rand()` 内部维护一个全局的种子状态，多个线程同时调用会产生数据竞争——这本身就违背了本章的原则。
2. **`rand_r()` 质量不佳**。它的可重入版本 `rand_r()` 虽然线程安全，但其随机数的统计质量很差，用于压力测试会引入偏差。

本实验改用 **PCG32** —— 一个小巧、高质量、可重入的随机数生成器。它的关键设计是：**每个线程拥有一个独立的生成器实例**，各自维护自己的状态，因此调用时**无需任何同步**：

```c
typedef struct {
  uint64_t state;   // 每次抽取都会推进
  uint64_t inc;     // 流选择器，恒为奇数
} pcg32_random_t;
```

`inc` 字段（流选择器）保证了：**用不同 `initseq` 初始化的两个生成器，永远不会产生相同的序列**。这正是给每个线程独立随机流的依据——各线程用自己的 rank 作为流编号，彼此独立，且结果可复现。

> 这是一个容易被忽视却很重要的工程细节：**压力测试工具本身绝不能引入额外的同步或竞争**，否则测的就不是被测对象了。

下面写入随机数模块的头文件与实现（内容取自 PCG 作者 M.E. O'Neill 的参考实现）。

In [ ]:
%%writefile {SRC_DIR}/pcg32_rand.h
#ifndef PCG32_RAND_H_
#define PCG32_RAND_H_

#include <stdint.h>

// PCG32 pseudo-random number generator.
// rand() is not thread-safe and rand_r() has poor statistical quality, so each
// thread owns a private pcg32_random_t instead. No synchronization is needed,
// and results stay reproducible for a given seed.
typedef struct {
  uint64_t state;  // advances on every draw
  uint64_t inc;    // stream selector, always odd
} pcg32_random_t;

// Initializes rng. initstate picks the starting point, initseq picks the
// stream: two generators with different initseq never produce the same
// sequence, which is what gives each thread an independent stream.
void pcg32_srandom_r(pcg32_random_t* rng, uint64_t initstate, uint64_t initseq);

// Returns a uniformly distributed 32-bit value.
uint32_t pcg32_random_r(pcg32_random_t* rng);

// Returns a double uniformly distributed in [0, 1).
double pcg32_rand_double(pcg32_random_t* rng);

#endif  // PCG32_RAND_H_

In [ ]:
%%writefile {SRC_DIR}/pcg32_rand.c
#include "pcg32_rand.h"

// LCG multiplier used by the reference PCG implementation.
#define PCG32_MULTIPLIER 6364136223846793005ULL

// Advances the state one step and returns the previous state.
static uint64_t pcg32_step(pcg32_random_t* rng) {
  uint64_t oldstate = rng->state;
  rng->state = oldstate * PCG32_MULTIPLIER + rng->inc;
  return oldstate;
}

void pcg32_srandom_r(pcg32_random_t* rng, uint64_t initstate,
                     uint64_t initseq) {
  rng->state = 0u;
  rng->inc = (initseq << 1u) | 1u;  // the increment must be odd
  pcg32_step(rng);
  rng->state += initstate;
  pcg32_step(rng);
}

uint32_t pcg32_random_r(pcg32_random_t* rng) {
  uint64_t oldstate = pcg32_step(rng);

  // XSH-RR output permutation: xorshift the high bits, then rotate right by an
  // amount taken from the top bits of the old state.
  uint32_t xorshifted = (uint32_t)(((oldstate >> 18u) ^ oldstate) >> 27u);
  uint32_t rot = (uint32_t)(oldstate >> 59u);
  return (xorshifted >> rot) | (xorshifted << ((-rot) & 31u));
}

double pcg32_rand_double(pcg32_random_t* rng) {
  return (double)pcg32_random_r(rng) / 4294967296.0;  // 2^32
}

## 5. 版本一 · 串行基准

先建立基准：单线程、无锁。它的作用有两方面：作为**正确性**的参照（其他版本的最终链表应与它一致），以及作为**性能**的参照（衡量并发版本是快是慢）。

三种操作的串行实现非常直接。以 `Insert` 为例，它遍历到正确位置后插入新节点：

```c
static int Insert(int value) {
  struct list_node_s *curr = head, *pred = NULL;
  while (curr != NULL && curr->data < value) {   // 找到插入位置
    pred = curr;
    curr = curr->next;
  }
  if (curr != NULL && curr->data == value) return 0;   // 已存在，不重复插入

  struct list_node_s *temp = malloc(sizeof(struct list_node_s));
  temp->data = value;
  temp->next = curr;
  if (pred == NULL) head = temp; else pred->next = temp;
  return 1;
}
```

程序先用固定种子预填充链表（保证四个版本面对**完全相同**的初始链表），再执行指定组成的随机操作，最后报告吞吐量。

In [ ]:
%%writefile {SRC_DIR}/pthread_link_list_base_seq.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#include "pcg32_rand.h"

#define MAX_KEY 100000000

struct list_node_s {
  int data;
  struct list_node_s* next;
};

// No lock is needed anywhere in this file: it is the single-threaded baseline
// against which the three concurrent versions are measured.
struct list_node_s* head = NULL;

int thread_count = 0;  // accepted only so the total work matches the parallel
                       // versions; the work itself runs on one thread
long ops_per_thread = 0;
double member_frac = 0.0;
double insert_frac = 0.0;
double delete_frac = 0.0;

long member_total = 0;
long insert_total = 0;
long delete_total = 0;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Inserts value into the sorted list. Returns 1 on success, 0 if it is already
// present.
static int Insert(int value) {
  struct list_node_s* curr = head;
  struct list_node_s* pred = NULL;

  while (curr != NULL && curr->data < value) {
    pred = curr;
    curr = curr->next;
  }

  if (curr != NULL && curr->data == value) {
    return 0;  // duplicate
  }

  struct list_node_s* temp = malloc(sizeof(struct list_node_s));
  if (temp == NULL) return 0;
  temp->data = value;
  temp->next = curr;

  if (pred == NULL) {
    head = temp;
  } else {
    pred->next = temp;
  }
  return 1;
}

// Returns 1 if value is present.
static int Member(int value) {
  struct list_node_s* curr = head;

  while (curr != NULL && curr->data < value) {
    curr = curr->next;
  }
  return (curr != NULL && curr->data == value);
}

// Removes value. Returns 1 on success, 0 if not found.
static int Delete(int value) {
  struct list_node_s* curr = head;
  struct list_node_s* pred = NULL;

  while (curr != NULL && curr->data < value) {
    pred = curr;
    curr = curr->next;
  }

  if (curr == NULL || curr->data != value) {
    return 0;  // not found
  }

  if (pred == NULL) {
    head = curr->next;
  } else {
    pred->next = curr->next;
  }
  free(curr);
  return 1;
}

static void Free_list(void) {
  struct list_node_s* curr = head;
  while (curr != NULL) {
    struct list_node_s* next = curr->next;
    free(curr);
    curr = next;
  }
  head = NULL;
}

int main(int argc, char* argv[]) {
  if (argc != 6) {
    fprintf(stderr,
            "Usage: %s <thread_count> <init_keys> <ops_per_thread> "
            "<member_frac> <insert_frac>\n",
            argv[0]);
    return 1;
  }

  thread_count = (int)strtol(argv[1], NULL, 10);
  long init_keys = strtol(argv[2], NULL, 10);
  ops_per_thread = strtol(argv[3], NULL, 10);
  member_frac = strtod(argv[4], NULL);
  insert_frac = strtod(argv[5], NULL);
  delete_frac = 1.0 - (member_frac + insert_frac);

  if (thread_count <= 0 || init_keys < 0 || ops_per_thread <= 0) {
    fprintf(stderr,
            "Error: thread_count and ops_per_thread must be positive\n");
    return 1;
  }
  if (member_frac < 0.0 || insert_frac < 0.0 || delete_frac < -1e-9) {
    fprintf(stderr, "Error: member_frac + insert_frac must not exceed 1.0\n");
    return 1;
  }

  long ops_total = (long)thread_count * ops_per_thread;

  printf("Concurrent Linked List - version 1: sequential baseline\n");
  printf("Init keys: %ld, Total ops: %ld\n", init_keys, ops_total);
  printf("Mix: member=%.2f insert=%.2f delete=%.2f\n\n", member_frac,
         insert_frac, delete_frac);

  // Pre-fill the list. A fixed seed keeps the list identical across all four
  // versions, so the comparison is fair.
  pcg32_random_t rng;
  pcg32_srandom_r(&rng, 42u, 54u);

  long inserted = 0;
  long attempts = 0;
  while (inserted < init_keys && attempts < 2L * init_keys) {
    int key = (int)(pcg32_random_r(&rng) % MAX_KEY);
    if (Insert(key)) ++inserted;
    ++attempts;
  }
  printf("Initial list size: %ld nodes\n", inserted);

  double start = get_time_ms();
  for (long i = 0; i < ops_total; ++i) {
    double which_op = pcg32_rand_double(&rng);
    int val = (int)(pcg32_random_r(&rng) % MAX_KEY);

    if (which_op < member_frac) {
      Member(val);
      ++member_total;
    } else if (which_op < member_frac + insert_frac) {
      Insert(val);
      ++insert_total;
    } else {
      Delete(val);
      ++delete_total;
    }
  }
  double elapsed = get_time_ms() - start;

  printf("\nTotal ops      : %ld\n", ops_total);
  printf("Member ops     : %ld\n", member_total);
  printf("Insert ops     : %ld\n", insert_total);
  printf("Delete ops     : %ld\n", delete_total);
  printf("Execution time : %.3f ms\n", elapsed);
  printf("Throughput     : %.0f ops/sec\n", ops_total / (elapsed / 1000.0));

  Free_list();
  return 0;
}

In [ ]:
seq = compile_c(f"{SRC_DIR}/pthread_link_list_base_seq",
                f"{SRC_DIR}/pthread_link_list_base_seq.c",
                f"{SRC_DIR}/pcg32_rand.c")
print()
# 参数：线程数 初始键数 每线程操作数 member比例 insert比例（delete=1-两者）
print("读多写少负载：4 线程等量工作，初始 1000 键，member=99%\n")
out = run_bin(seq, 4, 1000, 50000, 0.99, 0.005)

> **命令行参数**：`<线程数> <初始键数> <每线程操作数> <member比例> <insert比例>`，delete 比例为剩余部分。
> 串行版本只用一个线程执行全部工作，但接受相同的参数，以便与并行版本做等量对比。

## 6. 版本二 · 一把大锁

最简单的并发保护：整个链表一把互斥量，任何操作开始前加锁、结束后解锁。

```c
pthread_mutex_t list_mutex;

pthread_mutex_lock(&list_mutex);      // 临界区覆盖整个操作
if (which_op < member_frac)      Member(val);
else if (which_op < ...)         Insert(val);
else                             Delete(val);
pthread_mutex_unlock(&list_mutex);
```

**正确性**：无可挑剔。任一时刻只有一个线程能碰链表，绝不会出现「遍历途中节点被删」的问题。

**但并发度几乎为零**：临界区覆盖了**整个操作，包括完整的遍历**。这意味着，即使有 8 个线程、8 个核心，同一时刻也只有一个线程在真正工作，其余 7 个都阻塞在 `list_mutex` 上。

> 这是实验三「细粒度锁 vs 粗粒度锁」的另一面。那里粗粒度锁是**优点**（把 $n$ 次加锁降为 $t$ 次）；这里一把大锁却是**缺点**（把本可并发的遍历强行串行化）。
>
> 区别在于：π 估算的临界区只是一次加法，而这里的临界区是一整趟链表遍历。**临界区的大小才是关键**，「粗」与「细」本身并无绝对优劣。

统计计数用了一把**独立的** `count_mutex`，与链表锁分开——这样更新计数不必占用宝贵的链表锁，是「最小化临界区」的又一次应用。

In [ ]:
%%writefile {SRC_DIR}/pthread_link_list_one_mutex.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#include "pcg32_rand.h"

#define MAX_KEY 100000000
#define MAX_THREADS 64

struct list_node_s {
  int data;
  struct list_node_s *next;
};

struct list_node_s *head = NULL;

// One lock for the whole list. Simple and obviously correct, but it serializes
// every operation: only one thread can be inside the list at any time.
pthread_mutex_t list_mutex;

// A separate lock for the statistics, so that updating the counters does not
// have to hold the list lock.
pthread_mutex_t count_mutex;

int thread_count = 0;
long ops_per_thread = 0;
double member_frac = 0.0;
double insert_frac = 0.0;
double delete_frac = 0.0;

long member_total = 0;
long insert_total = 0;
long delete_total = 0;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// The three operations are the plain sequential ones: all synchronization is
// done by the caller, which holds list_mutex for the whole call.
static int Insert(int value) {
  struct list_node_s *curr = head;
  struct list_node_s *pred = NULL;

  while (curr != NULL && curr->data < value) {
    pred = curr;
    curr = curr->next;
  }
  if (curr != NULL && curr->data == value) return 0;

  struct list_node_s *temp = malloc(sizeof(struct list_node_s));
  if (temp == NULL) return 0;
  temp->data = value;
  temp->next = curr;

  if (pred == NULL) {
    head = temp;
  } else {
    pred->next = temp;
  }
  return 1;
}

static int Member(int value) {
  struct list_node_s *curr = head;
  while (curr != NULL && curr->data < value) {
    curr = curr->next;
  }
  return (curr != NULL && curr->data == value);
}

static int Delete(int value) {
  struct list_node_s *curr = head;
  struct list_node_s *pred = NULL;

  while (curr != NULL && curr->data < value) {
    pred = curr;
    curr = curr->next;
  }
  if (curr == NULL || curr->data != value) return 0;

  if (pred == NULL) {
    head = curr->next;
  } else {
    pred->next = curr->next;
  }
  free(curr);
  return 1;
}

static void Free_list(void) {
  struct list_node_s *curr = head;
  while (curr != NULL) {
    struct list_node_s *next = curr->next;
    free(curr);
    curr = next;
  }
  head = NULL;
}

void *Thread_work(void *rank) {
  long my_rank = (long)rank;

  // Each thread owns a private generator seeded on its own stream, so the
  // threads draw independent sequences without any locking.
  pcg32_random_t rng;
  pcg32_srandom_r(&rng, 1234u + (uint64_t)my_rank,
                  5678u + (uint64_t)(my_rank * 2));

  long my_member = 0, my_insert = 0, my_delete = 0;

  for (long i = 0; i < ops_per_thread; ++i) {
    double which_op = pcg32_rand_double(&rng);
    int val = (int)(pcg32_random_r(&rng) % MAX_KEY);

    // The critical section covers the entire operation, including the whole
    // traversal. This is what makes the version correct and slow.
    pthread_mutex_lock(&list_mutex);
    if (which_op < member_frac) {
      Member(val);
      ++my_member;
    } else if (which_op < member_frac + insert_frac) {
      Insert(val);
      ++my_insert;
    } else {
      Delete(val);
      ++my_delete;
    }
    pthread_mutex_unlock(&list_mutex);
  }

  // Accumulate locally, then merge once: the same coarse-grained idea used in
  // the pi experiment.
  pthread_mutex_lock(&count_mutex);
  member_total += my_member;
  insert_total += my_insert;
  delete_total += my_delete;
  pthread_mutex_unlock(&count_mutex);

  return NULL;
}

int main(int argc, char *argv[]) {
  if (argc != 6) {
    fprintf(stderr,
            "Usage: %s <thread_count> <init_keys> <ops_per_thread> "
            "<member_frac> <insert_frac>\n",
            argv[0]);
    return 1;
  }

  thread_count = (int)strtol(argv[1], NULL, 10);
  long init_keys = strtol(argv[2], NULL, 10);
  ops_per_thread = strtol(argv[3], NULL, 10);
  member_frac = strtod(argv[4], NULL);
  insert_frac = strtod(argv[5], NULL);
  delete_frac = 1.0 - (member_frac + insert_frac);

  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }
  if (init_keys < 0 || ops_per_thread <= 0) {
    fprintf(stderr, "Error: ops_per_thread must be positive\n");
    return 1;
  }
  if (member_frac < 0.0 || insert_frac < 0.0 || delete_frac < -1e-9) {
    fprintf(stderr, "Error: member_frac + insert_frac must not exceed 1.0\n");
    return 1;
  }

  pthread_mutex_init(&list_mutex, NULL);
  pthread_mutex_init(&count_mutex, NULL);

  printf("Concurrent Linked List - version 2: one global mutex\n");
  printf("Threads: %d, Init keys: %ld, Ops/thread: %ld\n", thread_count,
         init_keys, ops_per_thread);
  printf("Mix: member=%.2f insert=%.2f delete=%.2f\n\n", member_frac,
         insert_frac, delete_frac);

  // Pre-fill before any thread exists, so no lock is needed here.
  pcg32_random_t rng_main;
  pcg32_srandom_r(&rng_main, 42u, 54u);
  long inserted = 0;
  long attempts = 0;
  while (inserted < init_keys && attempts < 2L * init_keys) {
    int key = (int)(pcg32_random_r(&rng_main) % MAX_KEY);
    if (Insert(key)) ++inserted;
    ++attempts;
  }
  printf("Initial list size: %ld nodes\n", inserted);

  pthread_t *thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  double start = get_time_ms();
  for (long t = 0; t < thread_count; ++t) {
    if (pthread_create(&thread_handles[t], NULL, Thread_work, (void *)t) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (long t = 0; t < thread_count; ++t) pthread_join(thread_handles[t], NULL);
  double elapsed = get_time_ms() - start;

  long ops_total = (long)thread_count * ops_per_thread;
  printf("\nTotal ops      : %ld\n", ops_total);
  printf("Member ops     : %ld\n", member_total);
  printf("Insert ops     : %ld\n", insert_total);
  printf("Delete ops     : %ld\n", delete_total);
  printf("Execution time : %.3f ms\n", elapsed);
  printf("Throughput     : %.0f ops/sec\n", ops_total / (elapsed / 1000.0));

  Free_list();
  pthread_mutex_destroy(&list_mutex);
  pthread_mutex_destroy(&count_mutex);
  free(thread_handles);
  return 0;
}

In [ ]:
one = compile_c(f"{SRC_DIR}/pthread_link_list_one_mutex",
                f"{SRC_DIR}/pthread_link_list_one_mutex.c",
                f"{SRC_DIR}/pcg32_rand.c")
print()
print("同样的读多写少负载：\n")
out = run_bin(one, 4, 1000, 50000, 0.99, 0.005)

## 7. 版本三 · 交接锁（逐节点锁）

一把大锁的问题在于**锁的粒度太粗**。既然瓶颈是「整趟遍历都被锁住」，那就把锁的粒度细化到**单个节点**——每个节点自带一把互斥量：

```c
struct list_node_s {
  int data;
  struct list_node_s *next;
  pthread_mutex_t mutex;      // 每个节点一把锁
};
```

### 7.1 交接锁的加锁规则

遍历时采用**交接锁**（hand-over-hand locking，又称 lock coupling）：**先锁住下一个节点，再释放当前节点**——如同攀岩时「抓稳下一个把手，才松开当前的手」。

```c
// Member 的遍历核心
pthread_mutex_lock(&head_mutex);
curr = head;
if (curr != NULL) pthread_mutex_lock(&curr->mutex);
pthread_mutex_unlock(&head_mutex);

while (curr != NULL && curr->data < value) {
  struct list_node_s *next = curr->next;
  if (next != NULL) pthread_mutex_lock(&next->mutex);   // 先锁下一个
  pthread_mutex_unlock(&curr->mutex);                   // 再放开当前
  curr = next;
}
```

任一时刻，一个线程最多同时持有**两把相邻的锁**。这样多个线程可以像列车一样，在链表的不同位置同时前进——只要它们不追尾，就能真正并发。

### 7.2 为什么交接锁不会死锁

多个线程各持有两把锁，会不会死锁？**不会**，理由正是实验四的**资源分级**。

链表是有序遍历的，所有线程都**从头向尾**、按相同方向逐节点加锁。于是任何线程「持有节点 $i$、等待节点 $i+1$」时，它请求的锁在物理顺序上总是更靠后。加锁顺序构成一个与链表方向一致的**全局偏序**，等待环无法形成。

<!--
| 场景 | 全局加锁顺序 |
|---|---|
| 实验四 · 哲学家就餐 | 先取编号较小的叉子 |
| 实验六 · 生产者-消费者 | 先取信号量，再取互斥量 |
| **本实验 · 交接锁** | **按链表物理先后，从头向尾逐节点加锁** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">场景</th>
      <th style="text-align: left;">全局加锁顺序</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">实验四 · 哲学家就餐</td>
      <td style="text-align: left;">先取编号较小的叉子</td>
    </tr>
    <tr>
      <td style="text-align: left;">实验六 · 生产者-消费者</td>
      <td style="text-align: left;">先取信号量，再取互斥量</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>本实验 · 交接锁</strong></td>
      <td style="text-align: left;"><strong>按链表物理先后，从头向尾逐节点加锁</strong></td>
    </tr>
  </tbody>
</table>

三者是同一条规范在不同数据结构上的体现：**为所有锁定义全局一致的获取顺序**。

### 7.3 ⚠️ 交接锁的代价

交接锁提高了并发度，但代价高昂：**锁操作的次数正比于链表长度**。

一把大锁做一次 `Member`，只需 1 次加锁 + 1 次解锁；而交接锁遍历一条长度为 $L$ 的链表，最坏要做 $L$ 次加锁 + $L$ 次解锁。即便无锁的用户态开销只有几纳秒，乘以链表长度也会变得可观。

> **交接锁把「一把粗锁的等待」换成了「大量细锁的操作开销」。** 这笔交易是否划算，取决于链表有多长、以及并发冲突有多严重。
> 第 9 节将看到：链表越长，交接锁越吃亏。

In [ ]:
%%writefile {SRC_DIR}/pthread_link_list_node_mutex.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#include "pcg32_rand.h"

#define MAX_KEY 100000000
#define MAX_THREADS 64

// Every node carries its own lock, which protects that node's data and its
// next pointer.
struct list_node_s {
  int data;
  struct list_node_s *next;
  pthread_mutex_t mutex;
};

struct list_node_s *head = NULL;

// Protects the head pointer itself, which no node owns.
pthread_mutex_t head_mutex;

pthread_mutex_t count_mutex;

int thread_count = 0;
long ops_per_thread = 0;
double member_frac = 0.0;
double insert_frac = 0.0;
double delete_frac = 0.0;

long member_total = 0;
long insert_total = 0;
long delete_total = 0;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Hand-over-hand (lock coupling): the traversal always holds at least one
// lock, and it acquires the next node's lock BEFORE releasing the current
// one. That is what stops another thread from unlinking the node the
// traversal is standing on.
//
// The locks are always taken in list order, head_mutex first, so the
// acquisition order is globally consistent and no deadlock is possible.

static int Insert(int value) {
  struct list_node_s *pred = NULL;
  struct list_node_s *curr = NULL;
  int rv = 1;

  pthread_mutex_lock(&head_mutex);
  curr = head;
  if (curr != NULL) pthread_mutex_lock(&curr->mutex);

  while (curr != NULL && curr->data < value) {
    // Release whatever we were holding one step back, now that curr is locked.
    if (pred != NULL) {
      pthread_mutex_unlock(&pred->mutex);
    } else {
      pthread_mutex_unlock(&head_mutex);
    }
    pred = curr;
    curr = curr->next;
    if (curr != NULL) pthread_mutex_lock(&curr->mutex);
  }

  if (curr != NULL && curr->data == value) {
    rv = 0;  // duplicate
  } else {
    struct list_node_s *temp = malloc(sizeof(struct list_node_s));
    if (temp == NULL) {
      rv = 0;
    } else {
      pthread_mutex_init(&temp->mutex, NULL);
      temp->data = value;
      temp->next = curr;
      if (pred == NULL) {
        head = temp;  // still holding head_mutex here
      } else {
        pred->next = temp;
      }
    }
  }

  if (curr != NULL) pthread_mutex_unlock(&curr->mutex);
  if (pred != NULL) {
    pthread_mutex_unlock(&pred->mutex);
  } else {
    pthread_mutex_unlock(&head_mutex);
  }
  return rv;
}

static int Member(int value) {
  struct list_node_s *curr = NULL;
  int rv = 0;

  pthread_mutex_lock(&head_mutex);
  curr = head;
  if (curr != NULL) pthread_mutex_lock(&curr->mutex);
  // Member never modifies a link, so once it holds the first node it no longer
  // needs the head pointer to stay still.
  pthread_mutex_unlock(&head_mutex);

  while (curr != NULL && curr->data < value) {
    struct list_node_s *next = curr->next;
    if (next != NULL) pthread_mutex_lock(&next->mutex);
    pthread_mutex_unlock(&curr->mutex);
    curr = next;
  }

  if (curr != NULL) {
    rv = (curr->data == value);
    pthread_mutex_unlock(&curr->mutex);
  }
  return rv;
}

static int Delete(int value) {
  struct list_node_s *pred = NULL;
  struct list_node_s *curr = NULL;
  int rv = 1;

  pthread_mutex_lock(&head_mutex);
  curr = head;
  if (curr != NULL) pthread_mutex_lock(&curr->mutex);

  while (curr != NULL && curr->data < value) {
    if (pred != NULL) {
      pthread_mutex_unlock(&pred->mutex);
    } else {
      pthread_mutex_unlock(&head_mutex);
    }
    pred = curr;
    curr = curr->next;
    if (curr != NULL) pthread_mutex_lock(&curr->mutex);
  }

  if (curr == NULL || curr->data != value) {
    rv = 0;  // not found
    if (curr != NULL) pthread_mutex_unlock(&curr->mutex);
    if (pred != NULL) {
      pthread_mutex_unlock(&pred->mutex);
    } else {
      pthread_mutex_unlock(&head_mutex);
    }
    return rv;
  }

  // Unlink curr. We hold the locks of both pred and curr, so no other thread
  // can be reading pred->next, and no other thread can be standing on curr.
  // Therefore freeing curr here is safe.
  if (pred == NULL) {
    head = curr->next;
  } else {
    pred->next = curr->next;
  }

  pthread_mutex_unlock(&curr->mutex);
  pthread_mutex_destroy(&curr->mutex);
  free(curr);

  if (pred != NULL) {
    pthread_mutex_unlock(&pred->mutex);
  } else {
    pthread_mutex_unlock(&head_mutex);
  }
  return rv;
}

static void Free_list(void) {
  struct list_node_s *curr = head;
  while (curr != NULL) {
    struct list_node_s *next = curr->next;
    pthread_mutex_destroy(&curr->mutex);
    free(curr);
    curr = next;
  }
  head = NULL;
}

void *Thread_work(void *rank) {
  long my_rank = (long)rank;

  pcg32_random_t rng;
  pcg32_srandom_r(&rng, 1000u + (uint64_t)my_rank,
                  2000u + (uint64_t)(my_rank * 2));

  long my_member = 0, my_insert = 0, my_delete = 0;

  for (long i = 0; i < ops_per_thread; ++i) {
    double which_op = pcg32_rand_double(&rng);
    int val = (int)(pcg32_random_r(&rng) % MAX_KEY);

    // No lock is taken here: each operation locks the nodes it walks over.
    if (which_op < member_frac) {
      Member(val);
      ++my_member;
    } else if (which_op < member_frac + insert_frac) {
      Insert(val);
      ++my_insert;
    } else {
      Delete(val);
      ++my_delete;
    }
  }

  pthread_mutex_lock(&count_mutex);
  member_total += my_member;
  insert_total += my_insert;
  delete_total += my_delete;
  pthread_mutex_unlock(&count_mutex);

  return NULL;
}

int main(int argc, char *argv[]) {
  if (argc != 6) {
    fprintf(stderr,
            "Usage: %s <thread_count> <init_keys> <ops_per_thread> "
            "<member_frac> <insert_frac>\n",
            argv[0]);
    return 1;
  }

  thread_count = (int)strtol(argv[1], NULL, 10);
  long init_keys = strtol(argv[2], NULL, 10);
  ops_per_thread = strtol(argv[3], NULL, 10);
  member_frac = strtod(argv[4], NULL);
  insert_frac = strtod(argv[5], NULL);
  delete_frac = 1.0 - (member_frac + insert_frac);

  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }
  if (init_keys < 0 || ops_per_thread <= 0) {
    fprintf(stderr, "Error: ops_per_thread must be positive\n");
    return 1;
  }
  if (member_frac < 0.0 || insert_frac < 0.0 || delete_frac < -1e-9) {
    fprintf(stderr, "Error: member_frac + insert_frac must not exceed 1.0\n");
    return 1;
  }

  pthread_mutex_init(&head_mutex, NULL);
  pthread_mutex_init(&count_mutex, NULL);

  printf("Concurrent Linked List - version 3: hand-over-hand node locks\n");
  printf("Threads: %d, Init keys: %ld, Ops/thread: %ld\n", thread_count,
         init_keys, ops_per_thread);
  printf("Mix: member=%.2f insert=%.2f delete=%.2f\n\n", member_frac,
         insert_frac, delete_frac);

  pcg32_random_t rng_main;
  pcg32_srandom_r(&rng_main, 42u, 54u);
  long inserted = 0;
  long attempts = 0;
  while (inserted < init_keys && attempts < 2L * init_keys) {
    int key = (int)(pcg32_random_r(&rng_main) % MAX_KEY);
    if (Insert(key)) ++inserted;
    ++attempts;
  }
  printf("Initial list size: %ld nodes\n", inserted);

  pthread_t *thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  double start = get_time_ms();
  for (long t = 0; t < thread_count; ++t) {
    if (pthread_create(&thread_handles[t], NULL, Thread_work, (void *)t) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (long t = 0; t < thread_count; ++t) pthread_join(thread_handles[t], NULL);
  double elapsed = get_time_ms() - start;

  long ops_total = (long)thread_count * ops_per_thread;
  printf("\nTotal ops      : %ld\n", ops_total);
  printf("Member ops     : %ld\n", member_total);
  printf("Insert ops     : %ld\n", insert_total);
  printf("Delete ops     : %ld\n", delete_total);
  printf("Execution time : %.3f ms\n", elapsed);
  printf("Throughput     : %.0f ops/sec\n", ops_total / (elapsed / 1000.0));

  Free_list();
  pthread_mutex_destroy(&head_mutex);
  pthread_mutex_destroy(&count_mutex);
  free(thread_handles);
  return 0;
}

In [ ]:
node = compile_c(f"{SRC_DIR}/pthread_link_list_node_mutex",
                 f"{SRC_DIR}/pthread_link_list_node_mutex.c",
                 f"{SRC_DIR}/pcg32_rand.c")
print()
print("同样的读多写少负载：\n")
out = run_bin(node, 4, 1000, 50000, 0.99, 0.005)

> 若在**单核**环境运行，交接锁很可能是**最慢**的版本——因为它付出了大量锁操作开销，却换不到任何真正的并发（只有一个核心）。这一结果符合预期，交接锁的价值只在多核真正并发时才显现。

## 8. 版本四 · 读写锁

回到最重要的负载特征：**查找操作占绝大多数**。

一把大锁的浪费在于：它把 `Member` 也当作独占操作。可是**多个 `Member` 之间根本不冲突**——它们都只读链表，同时读毫无问题。真正需要独占的只有 `Insert` 和 `Delete`。

**读写锁**（read-write lock）正是为此而生：

```c
pthread_rwlock_t rwlock;

// Member —— 读操作，用读锁：多个读者可同时进入
pthread_rwlock_rdlock(&rwlock);
Member(val);
pthread_rwlock_unlock(&rwlock);

// Insert / Delete —— 写操作，用写锁：独占，排斥所有其他读者与写者
pthread_rwlock_wrlock(&rwlock);
Insert(val);
pthread_rwlock_unlock(&rwlock);
```

它的规则是：

<!--
| | 读锁 `rdlock` | 写锁 `wrlock` |
|---|---|---|
| 多个读者 | **可同时持有** | — |
| 读者与写者 | 互斥 | 互斥 |
| 多个写者 | — | 互斥 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">读锁 <code>rdlock</code></th>
      <th style="text-align: left;">写锁 <code>wrlock</code></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">多个读者</td>
      <td style="text-align: left;"><strong>可同时持有</strong></td>
      <td style="text-align: left;">—</td>
    </tr>
    <tr>
      <td style="text-align: left;">读者与写者</td>
      <td style="text-align: left;">互斥</td>
      <td style="text-align: left;">互斥</td>
    </tr>
    <tr>
      <td style="text-align: left;">多个写者</td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">互斥</td>
    </tr>
  </tbody>
</table>

一句话：**读者共享，写者独占**。

### 读写锁 API

```c
int pthread_rwlock_init(pthread_rwlock_t *rwlock, const pthread_rwlockattr_t *attr);
int pthread_rwlock_rdlock(pthread_rwlock_t *rwlock);   // 加读锁（共享）
int pthread_rwlock_wrlock(pthread_rwlock_t *rwlock);   // 加写锁（独占）
int pthread_rwlock_unlock(pthread_rwlock_t *rwlock);   // 读写锁统一用它解锁
int pthread_rwlock_destroy(pthread_rwlock_t *rwlock);
```

在 99% 都是 `Member` 的负载下，绝大多数操作都能获得读锁并**同时**进行，并发度接近理想。而它的实现远比交接锁简单——只有一把锁，没有逐节点的开销。

> ⚠️ **读写锁并非万能。** 它有两个需要注意的地方：
> 1. **写者可能饥饿**。读者源源不断时，写者可能长时间抢不到写锁（具体取决于实现的公平策略）。
> 2. **写多读少时不如普通互斥量**。维护读者计数本身有开销；若写操作频繁，读写锁反而更慢。

因此读写锁的适用前提非常明确：**读远多于写**。这正是本实验反复强调那条负载特征的原因。

In [ ]:
%%writefile {SRC_DIR}/pthread_link_list_rwlock.c
#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#include "pcg32_rand.h"

#define MAX_KEY 100000000
#define MAX_THREADS 64

struct list_node_s {
  int data;
  struct list_node_s *next;
};

struct list_node_s *head = NULL;

// One lock for the whole list, but it distinguishes readers from writers:
// Member takes it in shared mode, Insert and Delete in exclusive mode. The
// structure is identical to the one-mutex version; only the lock type changes.
pthread_rwlock_t rwlock;

pthread_mutex_t count_mutex;

int thread_count = 0;
long ops_per_thread = 0;
double member_frac = 0.0;
double insert_frac = 0.0;
double delete_frac = 0.0;

long member_total = 0;
long insert_total = 0;
long delete_total = 0;

static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Plain sequential operations again: the caller holds the appropriate lock.
static int Insert(int value) {
  struct list_node_s *curr = head;
  struct list_node_s *pred = NULL;

  while (curr != NULL && curr->data < value) {
    pred = curr;
    curr = curr->next;
  }
  if (curr != NULL && curr->data == value) return 0;

  struct list_node_s *temp = malloc(sizeof(struct list_node_s));
  if (temp == NULL) return 0;
  temp->data = value;
  temp->next = curr;

  if (pred == NULL) {
    head = temp;
  } else {
    pred->next = temp;
  }
  return 1;
}

static int Member(int value) {
  struct list_node_s *curr = head;
  while (curr != NULL && curr->data < value) {
    curr = curr->next;
  }
  return (curr != NULL && curr->data == value);
}

static int Delete(int value) {
  struct list_node_s *curr = head;
  struct list_node_s *pred = NULL;

  while (curr != NULL && curr->data < value) {
    pred = curr;
    curr = curr->next;
  }
  if (curr == NULL || curr->data != value) return 0;

  if (pred == NULL) {
    head = curr->next;
  } else {
    pred->next = curr->next;
  }
  free(curr);
  return 1;
}

static void Free_list(void) {
  struct list_node_s *curr = head;
  while (curr != NULL) {
    struct list_node_s *next = curr->next;
    free(curr);
    curr = next;
  }
  head = NULL;
}

void *Thread_work(void *rank) {
  long my_rank = (long)rank;

  pcg32_random_t rng;
  pcg32_srandom_r(&rng, 1000u + (uint64_t)my_rank,
                  2000u + (uint64_t)(my_rank * 2));

  long my_member = 0, my_insert = 0, my_delete = 0;

  for (long i = 0; i < ops_per_thread; ++i) {
    double which_op = pcg32_rand_double(&rng);
    int val = (int)(pcg32_random_r(&rng) % MAX_KEY);

    if (which_op < member_frac) {
      // Shared mode: any number of readers may traverse at the same time.
      pthread_rwlock_rdlock(&rwlock);
      Member(val);
      pthread_rwlock_unlock(&rwlock);
      ++my_member;
    } else {
      // Exclusive mode: no other reader or writer may be inside the list.
      pthread_rwlock_wrlock(&rwlock);
      if (which_op < member_frac + insert_frac) {
        Insert(val);
        ++my_insert;
      } else {
        Delete(val);
        ++my_delete;
      }
      pthread_rwlock_unlock(&rwlock);
    }
  }

  pthread_mutex_lock(&count_mutex);
  member_total += my_member;
  insert_total += my_insert;
  delete_total += my_delete;
  pthread_mutex_unlock(&count_mutex);

  return NULL;
}

int main(int argc, char *argv[]) {
  if (argc != 6) {
    fprintf(stderr,
            "Usage: %s <thread_count> <init_keys> <ops_per_thread> "
            "<member_frac> <insert_frac>\n",
            argv[0]);
    return 1;
  }

  thread_count = (int)strtol(argv[1], NULL, 10);
  long init_keys = strtol(argv[2], NULL, 10);
  ops_per_thread = strtol(argv[3], NULL, 10);
  member_frac = strtod(argv[4], NULL);
  insert_frac = strtod(argv[5], NULL);
  delete_frac = 1.0 - (member_frac + insert_frac);

  if (thread_count <= 0 || thread_count > MAX_THREADS) {
    fprintf(stderr, "Error: thread_count must be between 1 and %d\n",
            MAX_THREADS);
    return 1;
  }
  if (init_keys < 0 || ops_per_thread <= 0) {
    fprintf(stderr, "Error: ops_per_thread must be positive\n");
    return 1;
  }
  if (member_frac < 0.0 || insert_frac < 0.0 || delete_frac < -1e-9) {
    fprintf(stderr, "Error: member_frac + insert_frac must not exceed 1.0\n");
    return 1;
  }

  pthread_rwlock_init(&rwlock, NULL);
  pthread_mutex_init(&count_mutex, NULL);

  printf("Concurrent Linked List - version 4: read-write lock\n");
  printf("Threads: %d, Init keys: %ld, Ops/thread: %ld\n", thread_count,
         init_keys, ops_per_thread);
  printf("Mix: member=%.2f insert=%.2f delete=%.2f\n\n", member_frac,
         insert_frac, delete_frac);

  pcg32_random_t rng_main;
  pcg32_srandom_r(&rng_main, 42u, 54u);
  long inserted = 0;
  long attempts = 0;
  while (inserted < init_keys && attempts < 2L * init_keys) {
    int key = (int)(pcg32_random_r(&rng_main) % MAX_KEY);
    if (Insert(key)) ++inserted;
    ++attempts;
  }
  printf("Initial list size: %ld nodes\n", inserted);

  pthread_t *thread_handles = malloc(thread_count * sizeof(pthread_t));
  if (thread_handles == NULL) {
    fprintf(stderr, "Error: memory allocation failed\n");
    return 1;
  }

  double start = get_time_ms();
  for (long t = 0; t < thread_count; ++t) {
    if (pthread_create(&thread_handles[t], NULL, Thread_work, (void *)t) != 0) {
      fprintf(stderr, "Error: pthread_create failed\n");
      return 1;
    }
  }
  for (long t = 0; t < thread_count; ++t) pthread_join(thread_handles[t], NULL);
  double elapsed = get_time_ms() - start;

  long ops_total = (long)thread_count * ops_per_thread;
  printf("\nTotal ops      : %ld\n", ops_total);
  printf("Member ops     : %ld\n", member_total);
  printf("Insert ops     : %ld\n", insert_total);
  printf("Delete ops     : %ld\n", delete_total);
  printf("Execution time : %.3f ms\n", elapsed);
  printf("Throughput     : %.0f ops/sec\n", ops_total / (elapsed / 1000.0));

  Free_list();
  pthread_rwlock_destroy(&rwlock);
  pthread_mutex_destroy(&count_mutex);
  free(thread_handles);
  return 0;
}

In [ ]:
rw = compile_c(f"{SRC_DIR}/pthread_link_list_rwlock",
               f"{SRC_DIR}/pthread_link_list_rwlock.c",
               f"{SRC_DIR}/pcg32_rand.c")
print()
print("同样的读多写少负载：\n")
out = run_bin(rw, 4, 1000, 50000, 0.99, 0.005)

## 9. 性能对比：读写比例如何决定最优策略

四个版本处理**完全相同**的初始链表与操作序列，唯一的差别是锁策略。下面在两种负载下比较它们的吞吐量。

In [ ]:
import matplotlib.pyplot as plt

NT = max(2, min(4, os.cpu_count()))
KEYS, OPS = 1000, 40000
bins = {"串行基准": seq, "一把大锁": one, "交接锁": node, "读写锁": rw}

# 两种负载：读多写少 vs 写较多
loads = {"读多写少 (99% 查找)": (0.99, 0.005), "写较多 (50% 查找)": (0.50, 0.25)}

results = {}
for load_name, (mf, inf) in loads.items():
    results[load_name] = {}
    for v, b in bins.items():
        tp = parse_throughput(run_bin(b, NT, KEYS, OPS, mf, inf, echo=False))
        results[load_name][v] = tp / 1000.0  # 转为 kops/sec

print(f"线程数 {NT}，初始 {KEYS} 键，每线程 {OPS} 操作，{os.cpu_count()} 核")
print(f"（吞吐量单位：千次操作/秒 kops/s，越高越好）\n")
print(f"{'版本':<12}" + "".join(f"{ln:>20}" for ln in loads))
print("-" * 54)
for v in bins:
    print(f"{v:<12}" + "".join(f"{results[ln][v]:>20.1f}" for ln in loads))

# 图中一律使用英文标签，避免依赖中文字体
LABEL_EN = {
    "串行基准": "Serial baseline",
    "一把大锁": "One big lock",
    "交接锁": "Hand-over-hand",
    "读写锁": "Read-write lock",
}
LOAD_EN = {
    "读多写少 (99% 查找)": "Read-heavy (99% member)",
    "写较多 (50% 查找)": "Write-heavy (50% member)",
}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
colors = ["#9aa0a6", "#C7000B", "#E8833A", "#2E7D32"]
for ax, (ln, res) in zip(axes, results.items()):
    bars = ax.bar([LABEL_EN[k] for k in res], list(res.values()), color=colors)
    for bar, val in zip(bars, res.values()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.0f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_title(LOAD_EN[ln])
    ax.set_ylabel("Throughput (kops/s)")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=15)
fig.suptitle(f"Throughput of the four locking strategies "
             f"({NT} threads, {os.cpu_count()} cores)")
plt.tight_layout()
plt.show()

### 9.1 链表长度的影响

交接锁的锁操作次数正比于链表长度（7.3 节）。下面固定读多写少负载，改变初始链表大小，观察交接锁如何随链表变长而劣化。

In [ ]:
sizes = [100, 1000, 10000]
series = {"一把大锁": one, "交接锁": node, "读写锁": rw}
data = {k: [] for k in series}

print(f"读多写少负载，{NT} 线程，{os.cpu_count()} 核（吞吐量 kops/s）\n")
print(f"{'链表长度':>10}" + "".join(f"{k:>14}" for k in series))
print("-" * 52)
for sz in sizes:
    row = {}
    for k, b in series.items():
        tp = parse_throughput(run_bin(b, NT, sz, 20000, 0.99, 0.005, echo=False))
        row[k] = tp / 1000.0
        data[k].append(row[k])
    print(f"{sz:>10}" + "".join(f"{row[k]:>14.1f}" for k in series))

fig, ax = plt.subplots(figsize=(8, 4.2))
for (k, ys), c in zip(data.items(), ["#C7000B", "#E8833A", "#2E7D32"]):
    ax.plot(sizes, ys, "o-", label=LABEL_EN[k], color=c, lw=2, ms=7)
ax.set_xscale("log")
ax.set_xticks(sizes)
ax.set_xticklabels([str(s) for s in sizes])
ax.set_xlabel("Initial list length (log scale)")
ax.set_ylabel("Throughput (kops/s)")
ax.set_title(f"Effect of list length on the locking strategies "
             f"(read-heavy, {NT} threads)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n链表越长，交接锁每次遍历要做的加锁/解锁次数越多，其劣势越明显。")

### 9.2 结果解读

**① 读多写少时，读写锁通常最优。** 99% 的操作是 `Member`，它们持读锁并发进行，几乎不互相阻塞。读写锁实现简单、没有逐节点开销，在这种负载下最能发挥多核。

**② 一把大锁把并发度压到最低。** 无论多少线程、多少核心，同一时刻只有一个操作在进行。它是正确的，但基本得不到并行收益。

**③ 交接锁的表现高度依赖链表长度。** 链表短时，逐节点锁的开销尚可接受；链表长时，$O(L)$ 次的加锁解锁使它迅速劣化。交接锁真正的用武之地，是**长期存在大量写操作、且冲突集中在链表局部**的场景——此时它能让多个写者在不同位置并发修改，这是读写锁做不到的（写锁是全局独占的）。

**④ 写较多时，读写锁的优势收窄。** 写操作占比上升后，写锁的全局独占使读写锁退化得接近一把大锁，维护读者计数的额外开销甚至可能使它略慢。

### 🎓 没有普适最优的锁

<!--
| 负载特征 | 推荐策略 | 理由 |
|---|---|---|
| 读远多于写 | **读写锁** | 读者并发，实现简单 |
| 写多、且冲突局部化 | **交接锁** | 多个写者可在不同位置并发 |
| 链表很短，或操作稀疏 | **一把大锁** | 简单，锁开销可忽略 |
| 单线程 | **无锁** | 任何锁都是纯开销 |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">负载特征</th>
      <th style="text-align: left;">推荐策略</th>
      <th style="text-align: left;">理由</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">读远多于写</td>
      <td style="text-align: left;"><strong>读写锁</strong></td>
      <td style="text-align: left;">读者并发，实现简单</td>
    </tr>
    <tr>
      <td style="text-align: left;">写多、且冲突局部化</td>
      <td style="text-align: left;"><strong>交接锁</strong></td>
      <td style="text-align: left;">多个写者可在不同位置并发</td>
    </tr>
    <tr>
      <td style="text-align: left;">链表很短，或操作稀疏</td>
      <td style="text-align: left;"><strong>一把大锁</strong></td>
      <td style="text-align: left;">简单，锁开销可忽略</td>
    </tr>
    <tr>
      <td style="text-align: left;">单线程</td>
      <td style="text-align: left;"><strong>无锁</strong></td>
      <td style="text-align: left;">任何锁都是纯开销</td>
    </tr>
  </tbody>
</table>

> 选择锁策略的依据，是**负载特征与数据规模**——读写比例、链表长度、冲突的空间分布，而不是「哪种锁更高级」。
> 这与实验六的结论完全一致：**先测量负载，再选择工具。**

### 关于本机结果

本实验的性能结论**必须在多核平台上验证**。单核环境下所有版本都无法真正并发，交接锁与读写锁的优势无从体现，交接锁还会因锁操作开销显得最慢——这是环境所限，并非结论有误。请在华为鲲鹏多核平台上重跑第 9 节，并尝试把线程数设为核心数。

## 10. 结果分析

本实验把并发保护从规则数组推进到动态数据结构，建立了三项认识：

**① 动态结构的并发是「保护」而非「划分」。** 数组可以按下标分给线程，各写各的；链表的节点位置不固定、操作会改变结构，必须在操作进行时保护结构不被破坏。

**② 锁的粒度是一种权衡，没有普适最优。** 一把大锁简单但并发度低；交接锁并发度高但锁操作开销正比于链表长度；读写锁在读多写少时接近理想，但写多时退化。粒度选择取决于负载。

**③ 负载特征决定策略。** 同一个链表，读多写少时读写锁最优，写多且冲突局部化时交接锁最优。脱离负载谈锁策略的优劣没有意义。

这三点与本章反复出现的主线一以贯之：**并发设计的第一步永远是理解问题——数据如何被访问、读写比例如何、冲突集中在何处——然后才是选择工具。**

### 与全章的关联

- 交接锁不会死锁，靠的是实验四的**资源分级**（按链表物理顺序加锁）；
- 一把大锁 vs 细粒度锁的取舍，是实验三**锁粒度**讨论在动态结构上的延续；
- 「先测负载、再选工具」，则与实验六、实验七的性能结论完全一致。

本实验是本章同步工具的一次综合运用。

## 11. 🔧 动手练习

请修改代码、重新编译并运行，观察行为的变化：

1. 固定读多写少负载，把线程数依次设为 1、2、4、8、16，绘制四个版本的吞吐量曲线，指出各版本的可扩展性差异。
2. 把 member 比例依次设为 1.0、0.9、0.5、0.1，观察读写锁与一把大锁的吞吐量之比如何变化，找出读写锁不再占优的临界读写比。
3. 在交接锁版本中，故意把 `Member` 的加锁顺序改为「先放开当前，再锁下一个」，用多线程运行并说明会出现什么问题（提示：节点可能在两次加锁之间被删除）。
4. 为读写锁设置写者优先属性（`pthread_rwlockattr_setkind_np`），在读多写少负载下测量写操作的平均延迟变化，讨论读者优先与写者优先的取舍。
5. 统计交接锁版本在一次运行中实际执行的加锁次数（在加锁处加原子计数器），验证它是否正比于「操作数 × 平均遍历长度」。

## 12. 🤔 思考题

- 交接锁任一时刻最多持有两把相邻的锁。为什么必须是「先锁下一个、再放当前」，而不能「先放当前、再锁下一个」？后者会带来什么风险？
- 交接锁按链表从头到尾的顺序加锁，因而不会死锁。若链表支持双向遍历（有的线程从头向尾、有的从尾向头），交接锁还安全吗？应如何修改才能避免死锁？
- 读写锁在读多写少时最优，但存在写者饥饿的风险。请设计一种策略，在保持读者并发的同时，保证写者不会无限期等待。
- 一把大锁的 `Member` 也持有独占锁，这是它的主要浪费。除了读写锁，还有没有别的办法让多个 `Member` 并发，同时保证 `Insert`/`Delete` 的安全？（提示：了解一下 RCU——读-拷贝-更新。）
- 本实验用固定种子预填充链表，保证四个版本面对相同的初始数据。为什么这一点对性能对比的公平性很重要？若每个版本用不同的随机链表，可能得出什么错误结论？
- 交接锁让多个线程在链表不同位置并发操作。但若所有操作都集中查找/修改链表**头部**附近的少数几个节点，交接锁的并发优势还存在吗？这提示了什么样的数据分布假设？

## 13. 小结与后续

本实验通过同一个有序链表的四种锁策略，展示了并发数据结构设计中的粒度权衡：

<!--
| 版本 | 锁策略 | 并发度 | 锁操作次数 | 适用场景 |
|---|---|---|---|---|
| **一 · 串行基准** | 无 | — | 0 | 单线程 |
| **二 · 一把大锁** | 全局互斥量 | 最低 | 每操作 1 次 | 短链表、稀疏操作 |
| **三 · 交接锁** | 每节点互斥量 | 高 | 正比于链表长度 | 写多、冲突局部化 |
| **四 · 读写锁** | 一把读写锁 | 读多写少时接近理想 | 每操作 1 次 | **读远多于写** |
-->
<table style="text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">锁策略</th>
      <th style="text-align: left;">并发度</th>
      <th style="text-align: left;">锁操作次数</th>
      <th style="text-align: left;">适用场景</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>一 · 串行基准</strong></td>
      <td style="text-align: left;">无</td>
      <td style="text-align: left;">—</td>
      <td style="text-align: left;">0</td>
      <td style="text-align: left;">单线程</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>二 · 一把大锁</strong></td>
      <td style="text-align: left;">全局互斥量</td>
      <td style="text-align: left;">最低</td>
      <td style="text-align: left;">每操作 1 次</td>
      <td style="text-align: left;">短链表、稀疏操作</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>三 · 交接锁</strong></td>
      <td style="text-align: left;">每节点互斥量</td>
      <td style="text-align: left;">高</td>
      <td style="text-align: left;">正比于链表长度</td>
      <td style="text-align: left;">写多、冲突局部化</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>四 · 读写锁</strong></td>
      <td style="text-align: left;">一把读写锁</td>
      <td style="text-align: left;">读多写少时接近理想</td>
      <td style="text-align: left;">每操作 1 次</td>
      <td style="text-align: left;"><strong>读远多于写</strong></td>
    </tr>
  </tbody>
</table>

本实验也串联起了前面的多个知识点：资源分级（交接锁不死锁）、锁粒度（大锁 vs 细锁）、最小化临界区（独立的计数锁）、先测负载再选工具。

➡️ **后续内容：实验九 伪共享：当正确的程序依然很慢**。到目前为止，我们关注的都是**正确性**与**锁的粒度**。实验九将揭示一个更隐蔽的问题：一段**完全正确、甚至没有任何锁**的并行程序，可能仅仅因为两个线程的变量恰好落在**同一条缓存行**上，就慢到接近串行。这把讨论从「线程与锁」推进到「**缓存与硬件**」——理解它，才能写出不仅正确、而且真正高效的并行代码。这也是本章的最后一站：微架构感知。